In [ ]:
skriv ut all kod i ipnyb dokumentet

In [1]:
pip install nbformat

In [2]:
import nbformat

notebook_path = '/content/Textklassificering_02_större_modell_ipnyb.ipynb'

try:
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_content = nbformat.read(f, as_version=4)

    print(f"Code from '{notebook_path}':\n")
    for cell in notebook_content.cells:
        if cell.cell_type == 'code':
            print(cell.source)
            print("\n---\n") # Separator for better readability
except FileNotFoundError:
    print(f"Error: The notebook file '{notebook_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Code from '/content/Textklassificering_02_större_modell_ipnyb.ipynb':

# @title Lösning: Flera textfiler + Grafer + Legacy Keras
import os

# 1. INSTALLERA TENSORFLOWJS
# AXELS ANMÄRKNING. Efter första körningen kan du dektivera installen genom att
# sätta ett # först på rad 7. Men GLÖM INTE DET SEN
!pip install tensorflowjs

# 2. TVINGA KERAS 2 (Legacy Mode - VIKTIGT FÖR ATT UNDVIKA FEL)
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_KERAS"] = "1"

import tensorflow as tf
import tensorflowjs as tfjs
import numpy as np
import pandas as pd
import json
import shutil
import zipfile
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from google.colab import files

print(f"TensorFlow version: {tf.__version__}")
print()

---

# @title Lösning: Flera textfiler + Grafer + Legacy Keras
import os

# 1. INSTALLERA TENSORFLOWJS
!pip install tensorflowjs

# 2. TVINGA KERAS 2 (Legacy Mode

In [3]:
import pandas as pd

def load_data_from_file(filepath):
    texts = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                texts.append(parts[0])
                labels.append(parts[1])
            else:
                print(f"Skipping malformed line in {filepath}: {line.strip()}")
    return texts, labels

# Load data from dataset3.txt
texts_dataset3, labels_dataset3 = load_data_from_file('/content/dataset3.txt')

# Create a DataFrame from the loaded data
df_dataset3 = pd.DataFrame({'text': texts_dataset3, 'label': labels_dataset3})

print(f"Successfully loaded {len(df_dataset3)} entries from dataset3.txt")
display(df_dataset3.head())

Successfully loaded 600 entries from dataset3.txt


,text,label
0,Denna film framstod som mest utmärkt igår kväll,positiv
1,Skådespelet blev lite väl halvbra idag,neutral
2,Denna film upplevdes som välgjord,positiv
3,Dialogen upplevdes som ganska underbar,positiv
4,Regin var mest alldaglig från början till slut,neutral


In [4]:
# Import necessary libraries from the original notebook context
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Define parameters based on the original notebook
oov_tok = "<OOV>"
vocab_size = 5000
embedding_dim = 16
max_length = 120
trunc_type = 'post'
padding_type = 'post'

# Prepare texts and labels from df_dataset3
training_sentences = df_dataset3['text'].tolist()
training_labels = df_dataset3['label'].tolist()

# Initialize and fit tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(training_sentences)

word_index = tokenizer.word_index
print(f"Found {len(word_index)} unique tokens.")

# Create sequences and pad them
sequences = tokenizer.texts_to_sequences(training_sentences)
padded = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Map labels to numerical values
label_map = {'negativ': 0, 'neutral': 1, 'positiv': 2}
num_classes = len(label_map)

# Convert labels to numerical format
label_sequences = [label_map[label] for label in training_labels]

# Convert numerical labels to one-hot encoded format
training_labels_final = to_categorical(label_sequences, num_classes=num_classes)

print(f"Shape of padded sequences: {padded.shape}")
print(f"Shape of final labels: {training_labels_final.shape}")

Found 88 unique tokens.
Shape of padded sequences: (600, 120)
Shape of final labels: (600, 3)


In [5]:
# Build the model as defined in the original notebook
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax') # Use num_classes for output layer
])

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the model
num_epochs = 30 # Using a default value, adjust if needed
history = model.fit(padded, training_labels_final, epochs=num_epochs, verbose=1)

print("Model training complete.")

Epoch 1/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3350 - loss: 1.0994
Epoch 2/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3300 - loss: 1.0987
Epoch 3/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3333 - loss: 1.0981
Epoch 4/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3283 - loss: 1.0991
Epoch 5/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3550 - loss: 1.0980
Epoch 6/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3483 - loss: 1.0973
Epoch 7/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3750 - loss: 1.0973
Epoch 8/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3500 - loss: 1.0984
Epoch 9/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3500 - loss: 1.0975
Epoch 10/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3450 - loss: 1.0967
Epoch 11/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3700 - loss: 1.0962
Epoch 12/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4033 - lo